In [15]:
import pandas as pd

pd.set_option('display.max_columns', 50)
df = pd.read_csv('6_Survey/survey_raw_data.csv', delimiter=';')
df = df.iloc[:,6:56]
df = df.where(df >= 0)

In [16]:
from krippendorff import krippendorff_alpha, interval_metric

arr = df.dropna(thresh=49).fillna('*').to_numpy()
arr_str = arr.astype(str)  # converts all elements (numbers and '*') to strings
array = arr_str.tolist()

print("interval metric: %.3f" % krippendorff_alpha(array, interval_metric, missing_items='*'))

interval metric: 0.228


In [17]:
# Analysis per question
question_means = []
for i in range(10):
    mean = df.dropna(thresh=49).iloc[:,i*5:i*5+5].mean().mean()
    print(f'{i + 1}. Q{i+1}: {mean}/5.0')
    question_means.append(mean)

1. Q1: 4.771428571428571/5.0
2. Q2: 4.2857142857142865/5.0
3. Q3: 4.109523809523809/5.0
4. Q4: 4.857142857142857/5.0
5. Q5: 3.8857142857142852/5.0
6. Q6: 4.942857142857143/5.0
7. Q7: 4.885714285714286/5.0
8. Q8: 4.885714285714286/5.0
9. Q9: 3.742857142857143/5.0
10. Q10: 5.0/5.0


In [18]:
# Analysis per item
items = ['The explanation contains no hallucinations.', 'The explanation is convincing.', 'The whole generated text contributes to the explanation.', 'The explanation is well-structured.', 'The explanation has an appropriate length.']
item_means = []
for i in range(5):
    mean = df.dropna(thresh=49).iloc[:,i::5].mean().mean()
    print(f'{i + 1}. Item: {items[i]} Mean: {mean:.2f}/5.0')
    item_means.append(mean)
    

1. Item: The explanation contains no hallucinations. Mean: 4.50/5.0
2. Item: The explanation is convincing. Mean: 4.49/5.0
3. Item: The whole generated text contributes to the explanation. Mean: 4.67/5.0
4. Item: The explanation is well-structured. Mean: 4.67/5.0
5. Item: The explanation has an appropriate length. Mean: 4.36/5.0


In [20]:
import plotly.express as px
import plotly.io as io

x_labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10']

# color specific bars orange
colors = ['darkorange' if lbl in ('Q2', 'Q3', 'Q5') else '#1F77B4' for lbl in x_labels]

fig = px.bar(x=x_labels, y=question_means, range_y=[1, 5.2], labels={'x': 'Questions', 'y': 'Rating'}, template='simple_white', width=600)
fig.update_layout(bargap=0.3)
fig.update_traces(
    text=[f"{x:.1f}" for x in question_means],
    textposition='outside',
    marker_color=colors
)
fig.update_yaxes(tickfont=dict(size=14))

# Add horizontal line at y = 3
fig.add_shape(
    type='line',
    x0=-0.5,
    x1=len(x_labels) - 0.5,
    y0=4,
    y1=4,
    line=dict(color='red', width=2, dash='dash')
)
# fig.write_image('question_eval_plot.jpg', scale=3)
fig.show()